# ExecuTorch-Spyre — Part 2: Run on physical Spyre

Loads the `.pte` generated by **Part 1** (`01_generate_pte.ipynb`) and executes it
on the physical IBM Spyre via the ExecuTorch C++ runtime.

**Prerequisite:** Ensure you have run Part 1 and shut down its kernel (Kernel → Shut Down
Kernel in JupyterLab). The VFIO device file descriptor is released when that kernel process
exits — `free_runtime()` alone is not sufficient.

The demo passes when the runner prints `PASS: Spyre delegate matches CPU fp16 reference`.

## 1. Locate artifacts and verify prerequisites

Resolves `REPO_ROOT`, `work_home`, and `ARTIFACTS` (matching Part 1) and verifies that the exported `.pte`
and tensor files exist.

In [ ]:
import json, os, sys
from pathlib import Path

# Locate REPO_ROOT: try (in order):
#   1. EXECUTORCH_SPYRE_REPO_ROOT env var
#   2. Notebook file location (VS Code / JupyterLab expose __vsc_ipynb_file__)
#   3. Upward traversal from CWD

def _find_repo_root() -> Path:
    candidates = []
    # Notebook's own directory (VS Code sets __vsc_ipynb_file__; JupyterLab sets __session__)
    nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__session__")
    if nb_file:
        candidates.append(Path(nb_file).resolve().parent)
    # CWD fallback
    candidates.append(Path.cwd().resolve())
    for start in candidates:
        for directory in [start, *start.parents]:
            if (directory / "pyproject.toml").exists():
                return directory
    return candidates[0]  # will fail the assert below with a clear message

REPO_ROOT = Path(os.environ.get("EXECUTORCH_SPYRE_REPO_ROOT", "") or _find_repo_root()).resolve()
assert (REPO_ROOT / "pyproject.toml").exists(), (
    f"Repo not found at {REPO_ROOT}.\n"
    "Set the EXECUTORCH_SPYRE_REPO_ROOT environment variable to the executorch-spyre repo root, "
    "or open this notebook from inside the repo directory."
)

work_home = Path(os.environ.get("EXECUTORCH_SPYRE_WORK_HOME", REPO_ROOT / "work"))
ARTIFACTS = Path(os.environ.get("EXECUTORCH_SPYRE_ARTIFACTS", work_home / "artifacts"))

for name in ("spyre_matmul.pte", "a.bin", "b.bin", "ref.bin"):
    path = ARTIFACTS / name
    assert path.is_file() and path.stat().st_size > 0, (
        f"Missing artifact: {path}\nPlease run 01_generate_pte.ipynb first."
    )
    print(f"{name:20s} {path.stat().st_size:>10,} bytes")
print("Artifacts verified in:", ARTIFACTS)

## 2. Build or reuse the ExecuTorch-Spyre C++ runner

Reuses `{work_home}/build-spyre/spyre_runner` when available (built by a previous run).
Otherwise runs `build_device_runtime.sh` — a one-time build of ~10–20 minutes.

In [ ]:
RUN_BUILD = Path(os.environ.get("EXECUTORCH_SPYRE_RUN_BUILD", work_home / "build-spyre"))
RUNNER = RUN_BUILD / "spyre_runner"
build_script = REPO_ROOT / "scripts/build_device_runtime.sh"

if RUNNER.is_file():
    print("Reusing runner:", RUNNER)
else:
    print("Runner not found — starting the one-time device runtime build (~10–20 min).")
    local_et = REPO_ROOT / "third_party/executorch"
    et_src = str(local_et) if (local_et / "CMakeLists.txt").is_file() else str(work_home / "src/executorch")
    %env REPO={REPO_ROOT}
    %env ET_SRC={et_src}
    %env ET_INSTALL={os.environ.get('EXECUTORCH_SPYRE_ET_INSTALL', str(work_home / 'et-install'))}
    %env RUN_BUILD={RUN_BUILD}
    %env VENV={sys.prefix}
    %env HOME_DIR={work_home}
    !bash {build_script}

assert RUNNER.is_file() and os.access(RUNNER, os.X_OK), f"Runner not executable: {RUNNER}"
print("Runner ready:", RUNNER)

## 3. Run the `.pte` on the physical Spyre

The runner loads `spyre_matmul.pte`, supplies the exported inputs from `a.bin` / `b.bin`,
executes through `SpyreBackend` on `spyre:0` (with `FLEX_DEVICE=PF`), and compares the result with `ref.bin`.

Because the Part 1 kernel has been shut down, the physical VFIO device is available for exclusive access.

In [ ]:
%env FLEX_DEVICE=PF
%env PYTHONUTF8=1
!{RUNNER} {ARTIFACTS}

## Result

A successful run demonstrates both required phases:

1. **Preprocessing** (Part 1): `torch.export → Edge IR → SpyrePartitioner → SpyreBackend.preprocess → .pte`
2. **Execution** (Part 2): the ExecuTorch C++ runtime loaded that `.pte`, invoked
   `SpyreBackend` on `spyre:0`, and matched a deterministic CPU reference.

For a model-level follow-on, export your own model with `executorch-spyre-export`
(see `docs/setup-reference.md`, "Exporting any model").